由於我們可以預期在Spotify API上面以track_id搜尋歌曲資訊時，取得的應該都會是該取的最新一筆資料，沒有歷史資料。所以，可以思考為了不浪費記憶體或拉長爬蟲所需完成的時間，我們可以針對 “遍歷了每個週報的csv取出來的每份track_id清單”做第二次清理：
將連續周數的track_id list合併且去除重複項，不讓合併後的該份清單中，重複出現同一首track_id。


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive # 掛載專案雲端資料夾，該資料夾存放本專案的.ipynb、csv檔。
drive.mount('/content/drive')
raw_csv_dir = Path('<google_drive_path_to>/02_Data/01_Spotify_chart_raw/')

In [ ]:
def collect_track_ids_by_region(raw_csv_dir: str, region_keyword: str):
   track_id_set = set()

   for file in Path(raw_csv_dir).iterdir():
       if region_keyword in file.name:
           df = pd.read_csv(file)
           df["track_id"] = df["uri"].apply(lambda x: x.split(":")[-1])
           track_id_set.update(df["track_id"])

   return track_id_set

# 設定有哪些地區和關鍵字要搜尋
region_config = {
   "jp": "regional-jp-weekly",
   "kr": "regional-kr-weekly",
   "global": "regional-global-weekly"}
raw_csv_dir = "<google_drive_path_to>/02_Data/01_Spotify_chart_raw" # 路徑有錯，請幫忙修正～

region_track_id_sets = {}
# 呼叫上面的函式
for region, keyword in region_config.items():
   region_track_id_sets[region] = collect_track_ids_by_region(raw_csv_dir, keyword)
jp_track_id_set_in_3m = region_track_id_sets["jp"]
kr_track_id_set_in_3m = region_track_id_sets["kr"]
global_track_id_set_in_3m = region_track_id_sets["global"]
